# Version 27 — public LB 7141.14 → 7141.56

Lineage: **6425.11 task286 reach38** → Ricardo **[7120 LB]** + selective Frank 7116.79 overrides.

**Version 27** emits **7141.56**: **base714114 (7141.14)** plus one Frank711679 override on `task286`.


## Original task286 surgery (6425.11)

The first notebook in this lineage rewired `task286.onnx` from `reach_58` to `reach_38` and pruned the unused tail.


In [1]:
import sys
import zipfile
import subprocess
from pathlib import Path

try:
    import onnx
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnx==1.21.0'])
    import onnx

def compact_task286(raw, reach_index=38):
    model = onnx.load_model_from_string(raw)
    target = f'reach_{reach_index}'
    for node in model.graph.node:
        if list(node.output) == ['reach_bool']:
            node.input[0] = target
            break
    need = {out.name for out in model.graph.output}
    keep_rev = []
    for node in reversed(model.graph.node):
        if any(out in need for out in node.output):
            keep_rev.append(node)
            need.update(inp for inp in node.input if inp)
    keep_outputs = {tuple(node.output) for node in keep_rev}
    ordered = [node for node in model.graph.node if tuple(node.output) in keep_outputs]
    used_inputs = {inp for node in ordered for inp in node.input if inp}
    initializers = [init for init in model.graph.initializer if init.name in used_inputs]
    del model.graph.node[:]
    model.graph.node.extend(ordered)
    del model.graph.initializer[:]
    model.graph.initializer.extend(initializers)
    onnx.checker.check_model(model)
    return model.SerializeToString()

print('task286 reach-shortening surgery reference loaded')


task286 reach-shortening surgery reference loaded


## What changed in Version 27

| Item | Value |
|---|---|
| Previous notebook version | 26 at **7141.14** |
| Base | base714114 at **7141.14** (submission `53941008`) |
| Single change | replace `task286.onnx` from Frank711679 |
| Local gain vs base | 43912 |
| Public probe | **7141.56** |
| Submission ref | 53941907 |


## Current artifact (Version 27)

| Item | Value |
|---|---:|
| Public LB | 7141.56 |
| Submission ref | 53941907 |
| ONNX files | 400 |
| Zip bytes | 526,567 |
| SHA256 | `92e3588ff65d4eced4d1adef2220e57727118a58be539230a158b48345a4a61c` |


## Version changelog

            | Version | Change | Public LB | Submission ref | Notes |
            |---:|---|---:|---:|---|
            | 7 | core12 (+ Frank `task050` on core11) | 7125.30 | 53912538 | base before v8 |
| 8 | + Frank `task277` on core12 | 7125.68 | 53914152 | earlier |
| 9 | + Frank `task094` on core12 | 7125.90 | 53914158 | earlier |
| 10 | + Frank `task324` on core12 | 7125.95 | 53914144 | earlier |
| 11 | + Frank `task250` on core12 | 7125.97 | 53914146 | earlier |
| 12 | + Frank `task330` on core12 | 7126.02 | 53914139 | earlier |
| 13 | + Frank `task336` on core12 | 7126.03 | 53914149 | earlier |
| 15 | + Frank `task330` + `task324` on core12 | 7126.67 | 53914151 | earlier |
| 16 | + Frank `task336` on combo667 | 7127.40 | 53924937 | earlier |
| 17 | + Frank `task250` on combo667 | 7127.34 | 53924940 | earlier |
| 18 | + Frank `task094` on combo667 | 7127.27 | 53924950 | earlier |
| 19 | + Frank `task008` on combo667 | 7127.16 | 53924947 | earlier |
| 20 | + Frank `task202` on combo667 | 7126.84 | 53924943 | earlier (below 7126.90 bar) |
| 21 | + Frank `task336` + `task250` on combo667 | 7128.07 | 53924954 | earlier |
| 22 | + biohack `task323` + `task094` on base712807 | 7129.40 | 53940506 | earlier |
| 23 | + biohack `task008` + `task202` on base712940 | 7130.06 | 53940596 | earlier |
| 24 | + biohack `task286` on base713006 | 7130.48 | 53940776 | earlier |
| 25 | + biohack7141 `task108` on base713048 | 7131.10 | 53940852 | earlier |
| 26 | + Biohack7141 align (47 tasks) | 7141.14 | 53941008 | earlier |
| 27 | + Frank711679 `task286` on base714114 | 7141.56 | 53941907 | this version |


## External context: beyond public 7141.14

Version 26 matched the public Biohack7141 / Mark B / NeuroGolf00 bar at **7141.14** on every ONNX byte. Version 27 is the first step **above** that bar: Frank **7116.79** still had a cheaper `task286` locally, and the public probe confirmed **7141.56**.

## Why only `task286` from Frank 7116.79

After ver26, a full local scan of Frank 7116.79 on the **7141.14** base found **13** tasks with positive local cost gain. `task286` dominated (**43,912** local gain). Other candidates (`task023`, `task202`, `task094`, …) looked promising locally but were rejected on public LB.

| Probe on ver26 base | Tasks swapped | Public LB | Outcome |
|---|---|---:|---|
| `probe 714114 plus frank286` | `task286` | **7141.56** | **adopted → this version** |
| `probe 714114 plus frank023` | `task023` | 7126.26 | rejected |
| `probe 714114 plus frank286 023` | `task286+023` | 7126.68 | rejected |
| `probe 714114 plus frank286 023 202` | `task286+023+202` | 7126.85 | rejected |
| `probe 714114 plus frank top6` | `286,023,202,094,323,277` | 7128.56 | rejected |

Local cost alone is not enough past **7141.14**; only the single-task `task286` swap cleared the bar.

## Milestones on the path to 7141.56

| Version | Public LB | Key change |
|---:|---:|---|
| 26 | 7141.14 | align 47 tasks to Biohack7141 |
| **27** | **7141.56** | + Frank711679 `task286` on ver26 base |


## Artifact audit

- Embedded base64 matches submission `53941907`.
- Confirmed public LB: **7141.56**.
- Expected SHA256: `92e3588ff65d4eced4d1adef2220e57727118a58be539230a158b48345a4a61c`.


In [2]:
import base64
import hashlib
import zipfile
from pathlib import Path

OUT = Path('/kaggle/working/submission.zip')
EXPECTED_BYTES = 526567
EXPECTED_SHA256 = '92e3588ff65d4eced4d1adef2220e57727118a58be539230a158b48345a4a61c'
OBSERVED_PUBLIC_LB = '7141.56'
SUBMISSION_REF = '53941907'

B64 = ''.join([
    '...',
    'AA==',
])

payload = base64.b64decode(B64)
if len(payload) != EXPECTED_BYTES:
    raise ValueError(f'byte size mismatch: {len(payload)} != {EXPECTED_BYTES}')

sha = hashlib.sha256(payload).hexdigest()
if sha != EXPECTED_SHA256:
    raise ValueError(f'sha256 mismatch: {sha} != {EXPECTED_SHA256}')

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_bytes(payload)

with zipfile.ZipFile(OUT) as zf:
    names = sorted(name for name in zf.namelist() if name.endswith('.onnx'))
    if len(names) != 400:
        raise ValueError(f'expected 400 ONNX files, found {len(names)}')

print('wrote:', OUT)
print('onnx files:', len(names))
print('zip bytes:', OUT.stat().st_size)
print('sha256:', sha)
print('observed public LB:', OBSERVED_PUBLIC_LB)
print('submission ref:', SUBMISSION_REF)


wrote: /kaggle/working/submission.zip
onnx files: 400
zip bytes: 526567
sha256: 92e3588ff65d4eced4d1adef2220e57727118a58be539230a158b48345a4a61c
observed public LB: 7141.56
submission ref: 53941907
